In [1]:
import os
# 1. [关键] 必须设置缓存到数据盘 (50G硬盘保命设置)
os.environ["HF_HOME"] = "/root/autodl-tmp/hf_cache"
# 2. [关键] 关闭 HF_TRANSFER 加速 (解决 RuntimeError: no permits available)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
# 3. [关键] 关闭 Xet 加速 (解决 CAS service error)
os.environ["HF_HUB_DISABLE_XET"] = "1"
# 4. [关键] 使用国内镜像 (解决连接超时)
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import sys
sys.path.append("..")
from unsloth import FastLanguageModel
import pandas as pd
from evaluation.metrics import calculate_absa_metrics, calculate_absa_metrics_without_conflict
from tqdm import tqdm
import torch
import json
from src.templates import SYSTEM_PROMPT, USER_PROMPT
from src.utils.config_loader import load_config

cfg = load_config("../configs/config.yaml")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
MODEL_PATH = "unsloth/Qwen3-8B-unsloth-bnb-4bit"
max_seq_length = cfg.model_student.max_seq_length
load_in_4bit = cfg.model_student.load_in_4bit
dtype = cfg.model_student.dtype
trust_remote_code = cfg.model_student.trust_remote_code

In [3]:
print(f"⏳ Loading model from {MODEL_PATH}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    trust_remote_code = trust_remote_code, 
)

# 1. 强制左侧填充 (Decoder-only 模型必须)
tokenizer.padding_side = "left"

# 2. 解决 Pad Token 缺失问题 (Qwen/Llama 默认没有 Pad)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
# 3. 同步 ID
tokenizer.pad_token_id = tokenizer.eos_token_id

# 显式开启推理模式
FastLanguageModel.for_inference(model)

print("✅ Model loaded.")
print(f"✅ Tokenizer Configured: Padding Side = {tokenizer.padding_side} (Must be 'left')")
print(f"✅ Pad Token ID: {tokenizer.pad_token_id}")

⏳ Loading model from unsloth/Qwen3-8B-unsloth-bnb-4bit...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Model loaded.
✅ Tokenizer Configured: Padding Side = left (Must be 'left')
✅ Pad Token ID: 151645


In [4]:
def flatten_dataset(file_path):
    """
    读取嵌套的 JSONL 文件，将每个 Aspect Term 拆分成独立的测试样本。
    输入: 一行包含多个 aspectTerms
    输出: 多行，每行一个 text + 一个 term + 一个 polarity
    """
    flattened_data = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            text = row['text']
            
            # 提取 aspectTerms
            if 'aspectTerms' in row and row['aspectTerms']:
                for term_obj in row['aspectTerms']:
                    # 核心过滤逻辑
                    # 只要 polarity 是 conflict (不论大小写)，直接跳过
                    if term_obj['polarity'].lower() == 'conflict':
                        continue
                    
                    flattened_data.append({
                        "text": text,
                        "term": term_obj['term'],       # 提取 "term"
                        "polarity": term_obj['polarity'] # 提取 "polarity" (gold label)
                    })
            
            # [可选] 如果想测 aspectCategories，可以把下面注释打开
            if 'aspectCategories' in row and row['aspectCategories']:
                for cat_obj in row['aspectCategories']:
                    # 同样的过滤逻辑
                    if cat_obj['polarity'].lower() == 'conflict':
                        continue
                    
                    flattened_data.append({
                        "text": text,
                        "term": cat_obj['category'],     # 用 category 作为 aspect 输入
                        "polarity": cat_obj['polarity']
                    })

    # 此时 DataFrame 里只有 Positive, Negative, Neutral
    return pd.DataFrame(flattened_data)

In [5]:
print("⏳ Reading datasets...")
df_rest = flatten_dataset("../data/processed/test_rest_clean.jsonl")
df_lap = flatten_dataset("../data/processed/test_lap_clean.jsonl")
# 合并
df_test = pd.concat([df_rest, df_lap], ignore_index=True)

print(f"✅ Data Loaded.")
print(f"   - Restaurant Samples: {len(df_rest)}")
print(f"   - Laptop Samples:     {len(df_lap)}")
print(f"   - Total Eval Samples: {len(df_test)}")

⏳ Reading datasets...
✅ Data Loaded.
   - Restaurant Samples: 2093
   - Laptop Samples:     638
   - Total Eval Samples: 2731


In [7]:
# ==========================================
# ⚡️ 极速批量推理 (Batch Inference)
# ==========================================

# 1. 基础设置
tokenizer.padding_side = "left" # 形式上设一下
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# 2. 准备数据
BATCH_SIZE = 64
all_texts = df_test['text'].tolist()
all_terms = df_test['term'].tolist()
all_golds = df_test['polarity'].tolist()

predictions = []
raw_outputs = []

print(f"🚀 Starting Bulletproof Inference (Total: {len(df_test)}, Batch: {BATCH_SIZE})...")

# --- 定义手动左填充函数 (核武器) ---
def manual_left_pad(batch_input_ids, pad_token_id):
    """
    不管 Tokenizer 怎么想，我们物理上强制左填充
    """
    # 找出这一批里最长的序列
    max_len = max([len(ids) for ids in batch_input_ids])
    
    # 创建全 Pad 的空白画布
    batch_size = len(batch_input_ids)
    # shape: [batch, max_len]
    padded_ids = torch.full((batch_size, max_len), pad_token_id, dtype=torch.long, device="cuda")
    attention_mask = torch.zeros((batch_size, max_len), dtype=torch.long, device="cuda")
    
    for i, ids in enumerate(batch_input_ids):
        seq_len = len(ids)
        # 重点：把内容填到最右边！(ids 填入 -seq_len 到 最后)
        # 这样左边剩下的自然就是 Pad
        padded_ids[i, -seq_len:] = torch.tensor(ids, dtype=torch.long, device="cuda")
        attention_mask[i, -seq_len:] = 1 # 有内容的地方设为 1
        
    return padded_ids, attention_mask

# 3. 批量循环
for i in tqdm(range(0, len(df_test), BATCH_SIZE)):
    batch_texts = all_texts[i : i + BATCH_SIZE]
    batch_terms = all_terms[i : i + BATCH_SIZE]
    
    # --- A. 构造 Prompts ---
    batch_prompts = []
    for text, term in zip(batch_texts, batch_terms):
        user_content = USER_PROMPT.format(text=text, aspect=term)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ]
        prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        batch_prompts.append(prompt_str)

    # --- B. Tokenize (不 Padding!) ---
    # 关键点：这里 padding=False，我们要拿到原始的长短不一的列表
    raw_encodings = [tokenizer.encode(p, add_special_tokens=False) for p in batch_prompts]

    # --- C. 手动 Left Padding ---
    input_ids, attention_mask = manual_left_pad(raw_encodings, tokenizer.pad_token_id)

    # --- D. Generate ---
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=512,
            temperature=0.6,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id
        )

    # --- E. Decode & Parse ---
    # 只取新生成的部分 (从 prompt 长度之后开始截取)
    # 注意：因为是 Left Padding，prompt 长度等于 input_ids.shape[1]
    generated_ids = outputs[:, input_ids.shape[1]:]
    batch_responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    
    for res in batch_responses:
        raw_outputs.append(res)
        try:
            if "Final Sentiment:" in res:
                pred = res.split("Final Sentiment:")[-1].strip().lower().rstrip(".")
            else:
                pred = "parsing_error"
        except:
            pred = "error"
        predictions.append(pred)

🚀 Starting Bulletproof Inference (Total: 2731, Batch: 64)...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 43/43 [14:07<00:00, 19.70s/it]


In [7]:
from collections import Counter
# ==========================================
# ⚡️ Self-Consistency 批量推理 (Batch Inference)
# ==========================================

# 1. 基础设置
tokenizer.padding_side = "left" 
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# 2. 准备数据
# [注意] 因为 Self-Consistency 会把输出膨胀 num_paths 倍
# 建议把 BATCH_SIZE 调小 (例如 8 或 4)，防止显存爆炸
BATCH_SIZE = 16
NUM_PATHS = 5   # Self-Consistency 路径数 (5次投票)

all_texts = df_test['text'].tolist()
all_terms = df_test['term'].tolist()
all_golds = df_test['polarity'].tolist()

predictions = [] # 存放最终投票后的预测结果
raw_outputs_trace = [] # 存放所有路径的原始文本 (用于调试)

print(f"🚀 Starting Self-Consistency Inference (Total: {len(df_test)}, Batch: {BATCH_SIZE}, Paths: {NUM_PATHS})...")

# --- 辅助函数: 提取情感 ---
def extract_sentiment_label(text):
    """从模型输出中提取情感，提取失败返回 'neutral'"""
    try:
        if "Final Sentiment:" in text:
            # 提取冒号后的内容，转小写，去空格和标点
            return text.split("Final Sentiment:")[-1].strip().lower().rstrip(".")
        return "neutral" # 兜底策略
    except:
        return "error"

# --- 定义手动左填充函数 (保持不变) ---
def manual_left_pad(batch_input_ids, pad_token_id):
    max_len = max([len(ids) for ids in batch_input_ids])
    batch_size = len(batch_input_ids)
    padded_ids = torch.full((batch_size, max_len), pad_token_id, dtype=torch.long, device="cuda")
    attention_mask = torch.zeros((batch_size, max_len), dtype=torch.long, device="cuda")
    
    for i, ids in enumerate(batch_input_ids):
        seq_len = len(ids)
        padded_ids[i, -seq_len:] = torch.tensor(ids, dtype=torch.long, device="cuda")
        attention_mask[i, -seq_len:] = 1 
        
    return padded_ids, attention_mask

# 3. 批量循环
for i in tqdm(range(0, len(df_test), BATCH_SIZE)):
    batch_texts = all_texts[i : i + BATCH_SIZE]
    batch_terms = all_terms[i : i + BATCH_SIZE]
    
    # --- A. 构造 Prompts ---
    batch_prompts = []
    for text, term in zip(batch_texts, batch_terms):
        user_content = USER_PROMPT.format(text=text, aspect=term)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ]
        prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        batch_prompts.append(prompt_str)

    # --- B. Tokenize ---
    raw_encodings = [tokenizer.encode(p, add_special_tokens=False) for p in batch_prompts]

    # --- C. 手动 Left Padding ---
    input_ids, attention_mask = manual_left_pad(raw_encodings, tokenizer.pad_token_id)

    # --- D. Generate (Self-Consistency 核心) ---
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=512,
            
            # [关键修改] SC 参数
            do_sample=True,                 # 必须开启采样
            temperature=0.7,                # 增加多样性 (0.6-0.7 比较合适)
            top_p=0.9,                      # 核采样
            num_return_sequences=NUM_PATHS, # 每个样本生成 5 条
            
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id
        )

    # --- E. Decode & Vote ---
    # outputs 的维度: [batch_size * NUM_PATHS, total_seq_len]
    # 我们只需要新生成的部分
    generated_ids = outputs[:, input_ids.shape[1]:]
    batch_responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    
    # [关键] 处理 5 倍的输出结果
    # 此时 batch_responses 是一个扁平列表，长度为 batch_size * 5
    # 我们需要按 5 个一组进行切分，还原回每个样本
    
    current_batch_size = len(batch_prompts)
    
    for j in range(current_batch_size):
        # 计算切片范围
        start_idx = j * NUM_PATHS
        end_idx = start_idx + NUM_PATHS
        
        # 获取当前样本的 5 条推理路径
        paths = batch_responses[start_idx : end_idx]
        
        # 记录一下原始输出以便排查
        raw_outputs_trace.append(paths)
        
        # 1. 提取每条路径的情感标签
        votes = [extract_sentiment_label(p) for p in paths]
        
        # 2. 多数投票 (Majority Voting)
        # Counter.most_common(1) 返回 [('positive', 3)]
        vote_counter = Counter(votes)
        final_prediction, count = vote_counter.most_common(1)[0]
        
        # 3. 存入结果列表
        predictions.append(final_prediction)

# 4. 简单的完整性检查
assert len(predictions) == len(all_golds), f"预测数量 {len(predictions)} 与 真实标签数量 {len(all_golds)} 不一致！"

print("✅ Self-Consistency Inference Complete.")
print(f"Sample prediction: {predictions[:5]}")

🚀 Starting Self-Consistency Inference (Total: 2731, Batch: 16, Paths: 5)...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 171/171 [3:02:05<00:00, 63.89s/it]

✅ Self-Consistency Inference Complete.
Sample prediction: ['positive', 'positive', 'positive', 'positive', 'positive']


In [10]:
results_no_conflict = calculate_absa_metrics_without_conflict(all_golds, predictions)
print("\n" + "-"*50)
print("📈 Core Results (3-way, Positive/Neutral/Negative):")
print(f"Accuracy:         {results_no_conflict['accuracy']:.2%}")
print(f"Macro F1:         {results_no_conflict['macro_f1']:.2%}")
print(f"Parse Error Rate: {results_no_conflict['parse_error_rate']:.2%}")
print("="*50)

save_dir = "../EvalRes"
file_name = "evaluation_results_batch_qwen3_baseline.csv"
save_path = os.path.join(save_dir, file_name)

# 2. 如果文件夹不存在，自动创建它 (关键步骤)
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"📂 Created directory: {save_dir}")

# 3. 保存
df_test.to_csv(save_path, index=False)
print(f"✅ Results successfully saved to: {save_path}")


--------------------------------------------------
📈 Core Results (3-way, Positive/Neutral/Negative):
Accuracy:         17.61%
Macro F1:         25.06%
Parse Error Rate: 78.21%
✅ Results successfully saved to: ../EvalRes/evaluation_results_batch_qwen3_baseline.csv


In [9]:
from sklearn.metrics import classification_report

# 映射回字符串方便阅读
int_to_label = {0: "Negative", 1: "Neutral", 2: "Positive"}
y_true_labels = [int_to_label.get(x, "Error") for x in all_golds]
y_pred_labels = [int_to_label.get(x, "Error") for x in predictions] # 注意：predictions 此时可能是 str，需确保转回 int 或直接用 metrics 里的 int 列表

# 更好的方式：直接利用你 metrics.py 里生成的中间变量
# 或者我们手动转换一下现有的
def safe_map(lab):
    mapping = {"negative": 0, "neutral": 1, "positive": 2, "conflict": 3}
    return mapping.get(str(lab).lower().strip().rstrip("."), 999)

y_true_int = [safe_map(x) for x in all_golds]
y_pred_int = [safe_map(x) for x in predictions]

print(classification_report(
    y_true_int, 
    y_pred_int, 
    labels=[0, 1, 2],  # 只保留前三类
    target_names=["Negative", "Neutral", "Positive"],
    digits=4
))

report_dict_3way = classification_report(
    y_true_int, 
    y_pred_int, 
    labels=[0, 1, 2], 
    target_names=["Negative", "Neutral", "Positive"],
    output_dict=True 
)

              precision    recall  f1-score   support

    Negative     0.7458    0.0806    0.1455       546
     Neutral     0.5122    0.1830    0.2697       459
    Positive     0.9515    0.2045    0.3367      1726

   micro avg     0.8098    0.1761    0.2893      2731
   macro avg     0.7365    0.1560    0.2506      2731
weighted avg     0.8365    0.1761    0.2872      2731



In [11]:
# 转换为 DataFrame 并转置 (transpose)，让类别变成行索引
df_report_3way = pd.DataFrame(report_dict_3way).transpose()

# 写入 CSV
csv_3way_path = os.path.join(save_dir, "classification_report_qwen3_baseline.csv")
df_report_3way.to_csv(csv_3way_path, index=True)

print(f"✅ 3way-CSV report saved to: {csv_3way_path}")

✅ 3way-CSV report saved to: ../EvalRes/classification_report_qwen3_cot_finetuned_3way_dpo.csv


## DPO

In [13]:
# 1. 构造 DPO 的结果 DataFrame
df_dpo_save = df_test.copy() # 复制测试集原始数据 (text, term, polarity)
df_dpo_save['prediction'] = predictions # 把内存里的 DPO 预测结果填进去

# 2. 保存到硬盘 (带预测列！)
df_dpo_save.to_csv("../EvalRes/dpo_predictions_full.csv", index=False)
print("✅ DPO 预测结果已保存 (含 prediction 列)！")

✅ DPO 预测结果已保存 (含 prediction 列)！


In [14]:
from unsloth import FastLanguageModel
import torch
from tqdm import tqdm

# ============================
# 1. 加载 SFT Best Model
# ============================
# 请替换为你 SFT 阶段保存的 Best Checkpoint 路径
sft_model_path = "../outputs/qwen3_cot_finetuned_data_augmentation*0.7_LoraRank128" 

print(f"🔄 正在加载 SFT 模型: {sft_model_path} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = sft_model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# ============================
# 2. 极速推理 (Batch 32)
# ============================
# 这里不用 Self-Consistency，单次推理即可 (SFT 本身主要是用来纠偏)
# 如果想更准，也可以用 SC，但速度会慢 5 倍
BATCH_SIZE = 32 

all_texts = df_test['text'].tolist()
all_terms = df_test['term'].tolist()
sft_predictions = []

print("🚀 开始 SFT 模型重新推理...")

# 复用之前的 Manual Left Pad 函数
def manual_left_pad(batch_input_ids, pad_token_id):
    max_len = max([len(ids) for ids in batch_input_ids])
    batch_size = len(batch_input_ids)
    padded_ids = torch.full((batch_size, max_len), pad_token_id, dtype=torch.long, device="cuda")
    attention_mask = torch.zeros((batch_size, max_len), dtype=torch.long, device="cuda")
    for i, ids in enumerate(batch_input_ids):
        seq_len = len(ids)
        padded_ids[i, -seq_len:] = torch.tensor(ids, dtype=torch.long, device="cuda")
        attention_mask[i, -seq_len:] = 1 
    return padded_ids, attention_mask

for i in tqdm(range(0, len(df_test), BATCH_SIZE)):
    batch_texts = all_texts[i : i + BATCH_SIZE]
    batch_terms = all_terms[i : i + BATCH_SIZE]
    
    # 构造 Prompt
    batch_prompts = []
    for text, term in zip(batch_texts, batch_terms):
        user_content = USER_PROMPT.format(text=text, aspect=term)
        messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_content}]
        batch_prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    
    # Tokenize & Generate
    raw_encodings = [tokenizer.encode(p, add_special_tokens=False) for p in batch_prompts]
    input_ids, attention_mask = manual_left_pad(raw_encodings, tokenizer.pad_token_id)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=512,
            do_sample=False, # SFT 用贪婪解码即可，求稳
            pad_token_id=tokenizer.pad_token_id
        )
        
    # Decode & Parse
    generated_ids = outputs[:, input_ids.shape[1]:]
    batch_res = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    
    for res in batch_res:
        if "Final Sentiment:" in res:
            pred = res.split("Final Sentiment:")[-1].strip().lower().rstrip(".")
        else:
            pred = "neutral"
        sft_predictions.append(pred)

# 保存 SFT 结果
df_sft_save = df_test.copy()
df_sft_save['prediction'] = sft_predictions
df_sft_save.to_csv("../EvalRes/sft_predictions_full.csv", index=False)
print("✅ SFT 预测结果已保存！")

🔄 正在加载 SFT 模型: ../outputs/qwen3_cot_finetuned_data_augmentation*0.7_LoraRank128 ...
==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🚀 开始 SFT 模型重新推理...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 86/86 [34:29<00:00, 24.07s/it]

✅ SFT 预测结果已保存！


In [16]:
from sklearn.metrics import classification_report, accuracy_score

# ==========================================
# 1. 读取 SFT 和 DPO 的预测结果
# ==========================================
# 请确保这两个文件在之前的步骤中已经生成并保存了
try:
    df_dpo = pd.read_csv("../EvalRes/dpo_predictions_full.csv")
    df_sft = pd.read_csv("../EvalRes/sft_predictions_full.csv")
    print(f"✅ 成功加载 DPO 结果: {len(df_dpo)} 条")
    print(f"✅ 成功加载 SFT 结果: {len(df_sft)} 条")
except FileNotFoundError:
    print("❌ 错误：找不到预测文件，请先运行上面两步保存 SFT 和 DPO 的预测结果！")
    raise

# ==========================================
# 2. 执行最终融合策略 (The Grand Finale)
# ==========================================
final_preds = []
sources = []

# 确保索引对齐
assert len(df_dpo) == len(df_sft)

print("🚀 开始执行融合策略 (DPO Priority + SFT Correction)...")

for i in range(len(df_dpo)):
    # 提取预测结果并标准化
    dpo_p = str(df_dpo.loc[i, 'prediction']).strip().lower()
    sft_p = str(df_sft.loc[i, 'prediction']).strip().lower()
    
    # --- 核心融合逻辑 ---
    
    # 1. DPO 的绝对领域：Positive / Negative
    # DPO 在这两类上 Precision 极高 (95%+)，如果它敢说是，我们就敢信
    if dpo_p in ['positive', 'negative']:
        final_preds.append(dpo_p)
        sources.append("DPO_Trust")
        
    # 2. DPO 的盲区：Neutral
    # DPO 倾向于过度保守把 Positive 判成 Neutral。
    # 如果 DPO 说 Neutral，我们要查 SFT 的票。
    else: # dpo_p == 'neutral' or others
        if sft_p in ['positive', 'negative']:
            # SFT 比较激进，如果 SFT 觉得是好评/差评，大概率 DPO 是过于谨慎了
            final_preds.append(sft_p)
            sources.append("SFT_Correction")
        else:
            # 两个都说是 Neutral，或者 SFT 也没看出名堂
            final_preds.append("neutral")
            sources.append("Consensus_Neutral")

# ==========================================
# 3. 计算最终指标
# ==========================================
# 获取真实标签
golds = df_dpo['polarity'].str.strip().str.lower().tolist()

print("\n" + "="*50)
print("🏆 最终融合模型成绩 (FINAL ENSEMBLE SCORE)")
print("="*50)

# 打印详细报告
report = classification_report(golds, final_preds, digits=4)
print(report)

acc = accuracy_score(golds, final_preds)
print(f"✨ Final Accuracy: {acc:.4%}")

# 统计决策来源
print("\n🔍 决策来源统计:")
print(pd.Series(sources).value_counts())

# ==========================================
# 4. 保存最终结果
# ==========================================
df_final = df_dpo.copy()
df_final['final_prediction'] = final_preds
df_final['sft_prediction'] = df_sft['prediction'] # 把 SFT 的也存进去方便对比
df_final['dpo_prediction'] = df_dpo['prediction'] # 把 DPO 的也存进去
df_final['source'] = sources # 记录决策来源

save_path = "../EvalRes/final_ensemble_submission.csv"
df_final.to_csv(save_path, index=False)
print(f"\n✅ 最终结果已保存至: {save_path}")
print("🎉 实验结束！祝贺你完成了从 SFT 到 DPO 再到 Ensemble 的完整优化闭环！")

✅ 成功加载 DPO 结果: 2731 条
✅ 成功加载 SFT 结果: 2731 条
🚀 开始执行融合策略 (DPO Priority + SFT Correction)...

🏆 最终融合模型成绩 (FINAL ENSEMBLE SCORE)
              precision    recall  f1-score   support

    negative     0.8272    0.8590    0.8428       546
     neutral     0.6166    0.5817    0.5987       459
    positive     0.9255    0.9282    0.9268      1726

    accuracy                         0.8561      2731
   macro avg     0.7898    0.7896    0.7894      2731
weighted avg     0.8539    0.8561    0.8549      2731

✨ Final Accuracy: 85.6097%

🔍 决策来源统计:
DPO_Trust            2084
Consensus_Neutral     433
SFT_Correction        214
Name: count, dtype: int64

✅ 最终结果已保存至: ../EvalRes/final_ensemble_submission.csv
🎉 实验结束！祝贺你完成了从 SFT 到 DPO 再到 Ensemble 的完整优化闭环！
